In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 2.3 — Identifying Key Features

**Objectives**

By the end of this section you will be able to:

- Compute and interpret a correlation matrix.
- Identify low-variance features that carry little predictive information.
- Use Random Forest feature importance to rank predictor variables.
- Select a focused subset of features to simplify and improve a model.

> **Cooking analogy:** Not every ingredient in your pantry belongs in tonight's
> dish. A skilled chef selects only the ingredients that meaningfully contribute
> to the flavour — too many competing ingredients muddle the taste and increase
> prep time. **Feature selection** works the same way: identify the variables
> that actually drive the outcome and leave the rest on the shelf.

**Feature selection** identifies which variables (features) are most important
for predicting an outcome. Fewer, more relevant features produce faster,
more interpretable models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(123)
n = 500

df_cust = pd.DataFrame({
    "Age"             : np.random.randint(18, 70, n),
    "Income"          : np.random.normal(60_000, 25_000, n).clip(20_000, 200_000),
    "Tenure_Months"   : np.random.randint(1, 120, n),
    "Num_Products"    : np.random.randint(1, 8, n),
    "Web_Visits"      : np.random.randint(0, 50, n),
    "Complaints"      : np.random.poisson(0.5, n),
    "Satisfaction"    : np.random.uniform(1, 10, n).round(1),
})

# Target: will the customer churn? (influenced by satisfaction and complaints)
df_cust["Churned"] = (
    (df_cust["Satisfaction"] < 5) |
    (df_cust["Complaints"]   > 2)
).astype(int)

print("Dataset shape:", df_cust.shape)
print("Churn rate: {:.1%}".format(df_cust["Churned"].mean()))

### Correlation Analysis

In [ ]:
corr_matrix = df_cust.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Focus on correlation with the target variable
target_corr = corr_matrix["Churned"].drop("Churned").sort_values(key=abs, ascending=False)
print("Correlation with Churn (sorted by strength):")
print(target_corr.round(3).to_string())

### Variance Analysis

Features with near-zero variance carry little information.

In [ ]:
feature_variance = df_cust.drop(columns="Churned").var().sort_values(ascending=False)
print("Feature Variance:")
print(feature_variance.round(2).to_string())

### Feature Importance via Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_cust.drop(columns="Churned")
y = df_cust["Churned"]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importance_df = pd.DataFrame({
    "Feature"   : X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance_df["Feature"], importance_df["Importance"], color="teal")
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importance (Random Forest)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
print("Top 3 most important features:")
print(importance_df.head(3).to_string(index=False))

---

### Student Task 2.3

Using `df_cust`:

1. Identify all features that have an **absolute correlation > 0.3** with `Churned`.
2. Create a bar chart showing the correlation of each feature with `Churned`.
3. Based on the Random Forest importance plot, which **two** features would you prioritise for a churn prediction model? Justify your choice.
4. What does it mean if a feature has a **negative** correlation with churn?

In [ ]:
# Your code here

---

### Evaluation Questions 2.3

1. A correlation of −0.75 between two variables indicates:
   a) No relationship
   b) A weak positive relationship
   c) A strong positive relationship
   d) A strong negative relationship (correct)

2. Feature importance from a Random Forest measures:
   a) How large a feature's values are
   b) How much each feature reduces prediction error (correct)
   c) The correlation between a feature and the target
   d) The number of unique values in a feature

3. A feature with near-zero variance should likely be:
   a) Normalised before use
   b) Kept as the primary predictor
   c) Removed — it carries little information (correct)
   d) Multiplied by the target variable

4. Why is feature selection important in business ML models?
   a) It always improves accuracy significantly
   b) It reduces model complexity and improves interpretability (correct)
   c) It automatically handles missing values
   d) It is required by all ML algorithms

5. Which seaborn function creates a correlation heatmap?
   a) `sns.corrplot()`
   b) `sns.matrix()`
   c) `sns.heatmap()` (correct)
   d) `sns.pairplot()`

---